# AirCasting: download recordings for AIRBEAM3:B0B21C7627C4

**Configured for your device.** Run section 2b first to test access. Discovery searches several spellings of the device ID, because the server matches package names exactly, and also downloads session 21374 from your map link.

**Run the cells from top to bottom.** This notebook uses the Python standard library; the downloaded Ruby/Rails application does not need to run. Python 3.10+ is recommended. Optional tables and maps use pandas and folium when installed.

1. Enter your full `sensor_package_name` strings in the configuration cell, for example the **format** `AirBeam:YOUR_DEVICE_IDENTIFIER`. A PM2.5 channel name or a session number is a different identifier.
2. If you do not know these strings, enter your AirCasting username in `LOOKUP_USERNAME`, run the optional lookup, and copy the returned identifiers into `DEVICE_PACKAGES`.
3. Choose dates. Leave both region settings empty for all locations, or use a rectangle / WGS84 GeoJSON polygon.
4. Run discovery, then download. Open the displayed CSV/ZIP links.

Access is to recordings already available through AirCasting, not directly to the physical devices. Use only your project's devices or authorised project tags. No credentials are needed by the public endpoints implemented here. Account-only retrieval is described separately at the end.

**Validation:** API behaviour was checked against the supplied `AirCasting-master` source. The default empty-configuration run and 21 offline checks passed, covering retrieval, CSV counts, regions, and time handling. Network access was unavailable during preparation; deployment compatibility and access to your devices still need to be checked when you run it. No example measurements are presented as your data.


## 1. Configuration

`DISCOVERY_START` must predate your earliest deployment. The v3 endpoint filters **session start dates**, so keep this earlier than `DOWNLOAD_FROM` to find long-running fixed sessions. `DOWNLOAD_UNTIL` is exclusive; the default includes today. Date filtering uses the source clock, not an assumed UTC conversion.

Leave `TIME_CONVENTION="unverified"` initially. The source describes fixed timestamps as local time encoded as UTC. After confirming your data convention, choose `local_as_utc` with the correct timezone, or `utc`. Original timestamps are always retained; ambiguous daylight-saving times are flagged. On Windows, timezone conversion may require `%pip install tzdata`; the default unverified mode does not require it.


In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone

BASE_URL = "https://aircasting.org"
MY_DEVICE_ID = "AIRBEAM3:B0B21C7627C4"   # your device, as written on the label/app

def package_variants(device_id):
    """AirCasting stores package names with exact spelling, so try the common forms."""
    model, _, mac = device_id.replace("-", ":").partition(":")
    model = "AirBeam" + model.upper().replace("AIRBEAM", "")
    variants = []
    for sep in (":", "-"):
        for m in (mac.lower(), mac.upper()):
            candidate = model + sep + m
            if candidate not in variants:
                variants.append(candidate)
    if device_id not in variants:
        variants.append(device_id)
    return variants

DEVICE_PACKAGES = package_variants(MY_DEVICE_ID)
KNOWN_SESSION_IDS = [21374]  # session from your map link; set [] if it is not this device
PROJECT_TAGS = []          # optional authorised project tags; OR within this list
LOOKUP_USERNAME = ""      # optional helper for finding mobile PM2.5 device identifiers

DISCOVERY_START = "2020-01-01"  # AirBeam3 was released in 2020; set just before your deployment
DOWNLOAD_FROM = "2020-01-01"
DOWNLOAD_UNTIL = (datetime.now() + timedelta(days=1)).date().isoformat()
DISCOVERY_WINDOW_DAYS = 365
FIXED_WINDOW_DAYS = 7

REGION_NAME = "all_locations"
BBOX = None               # (west, south, east, north); coordinates in degrees
# Example rectangle around Donegal, NOT an administrative boundary:
# BBOX = (-8.9, 54.4, -6.9, 55.5)
REGION_GEOJSON = None     # e.g. "my_region.geojson"; Polygon/MultiPolygon in WGS84

TIME_CONVENTION = "unverified"   # "unverified", "local_as_utc", or "utc"
TIMEZONE_NAME = "Europe/Dublin"
OUTPUT_ROOT = Path("aircasting_downloads")
REFRESH = True            # True checks for new/changed uploads; False resumes cached requests
REQUEST_TIMEOUT = 45
REQUEST_PAUSE = 0.2

print("Device ID variants that will be searched:")
for p in DEVICE_PACKAGES:
    print("  ", p)


Device ID variants that will be searched:
   AirBeam3:b0b21c7627c4
   AirBeam3:B0B21C7627C4
   AirBeam3-b0b21c7627c4
   AirBeam3-B0B21C7627C4
   AIRBEAM3:B0B21C7627C4


## 2. API client and file helpers

Only GET requests are used. Responses are cached locally with their retrieval time. Reruns write new export snapshots instead of appending duplicate rows. `REFRESH=False` reuses matching cached requests and can resume an interrupted download; switch back to `True` to check for newly uploaded or corrected data. Cache and exported files can contain precise locations: keep this folder private.

The source has no pagination parameters for `/api/v3/sessions`; discovery is split into explicit date windows. The optional older mobile-search endpoint does use `limit` and `offset`.


In [2]:
import base64, csv, gzip, hashlib, io, json, math, os, re, time, zipfile
from collections import Counter
from urllib.parse import urlencode, urlsplit, quote
from urllib.request import Request, build_opener, HTTPRedirectHandler
from urllib.error import HTTPError, URLError
from zoneinfo import ZoneInfo

class NoRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

def safe_name(value):
    text = str(value)
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)[:75] + "_" + hashlib.sha256(text.encode()).hexdigest()[:8]

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

class AirCastingClient:
    def __init__(self, root=OUTPUT_ROOT):
        parsed = urlsplit(BASE_URL)
        if parsed.scheme != "https" or parsed.hostname not in {"aircasting.org", "aircasting.habitatmap.org"} or parsed.username or parsed.password:
            raise ValueError("Use the documented HTTPS AirCasting host, without credentials in the URL.")
        self.root = Path(root)
        self.cache = self.root / "raw_responses"
        self.cache.mkdir(parents=True, exist_ok=True)
        self.opener = build_opener(NoRedirect())
        self.calls = []

    def get(self, path, params=None, token=None, refresh=None):
        if not path.startswith("/api/") or ".." in path:
            raise ValueError("Expected an AirCasting API path")
        params = params or {}
        url = BASE_URL.rstrip("/") + path + ("?" + urlencode(params, doseq=True) if params else "")
        cache_path = self.cache / (hashlib.sha256(url.encode()).hexdigest() + ".json.gz")
        refresh = REFRESH if refresh is None else refresh
        if not token and not refresh and cache_path.exists():
            with gzip.open(cache_path, "rt", encoding="utf-8") as fh:
                envelope = json.load(fh)
            self.calls.append({"path": path, "cache": str(cache_path), "cached": True, "fetched_at": envelope["fetched_at"]})
            return envelope["data"]
        headers = {"Accept": "application/json", "User-Agent": "AirCastingResearchNotebook/1.0"}
        if token:
            headers["Authorization"] = "Basic " + base64.b64encode((token + ":X").encode()).decode()
        for attempt in range(4):
            try:
                time.sleep(REQUEST_PAUSE)
                with self.opener.open(Request(url, headers=headers, method="GET"), timeout=REQUEST_TIMEOUT) as response:
                    content_type = response.headers.get("Content-Type", "")
                    if "json" not in content_type.lower():
                        raise RuntimeError("The endpoint returned non-JSON content; check deployed API compatibility.")
                    raw = response.read()
                    if response.headers.get("Content-Encoding") == "gzip":
                        raw = gzip.decompress(raw)
                    data = json.loads(raw)
                fetched_at = datetime.now(timezone.utc).isoformat()
                if not token:
                    temporary = cache_path.with_name(cache_path.name + ".tmp")
                    with gzip.open(temporary, "wt", encoding="utf-8") as fh:
                        json.dump({"endpoint": path, "params": params, "fetched_at": fetched_at, "data": data}, fh, ensure_ascii=False)
                    temporary.replace(cache_path)
                self.calls.append({"path": path, "cache": str(cache_path) if not token else None, "cached": False, "fetched_at": fetched_at})
                return data
            except HTTPError as error:
                if error.code not in {429, 500, 502, 503, 504} or attempt == 3:
                    raise RuntimeError(f"HTTP {error.code} at {path}. Check access, identifiers, and deployed API version.") from None
                retry = error.headers.get("Retry-After", "")
                delay = float(retry) if retry.isdigit() else 2 ** attempt
                if delay > 60:
                    raise RuntimeError(f"Server requested a {delay:g}-second pause. Retry later; cached responses are retained.") from None
                time.sleep(delay)
            except (URLError, TimeoutError) as error:
                if attempt == 3:
                    raise RuntimeError(f"Network request failed at {path}; check your connection.") from None
                time.sleep(2 ** attempt)

def as_clock(value):
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value / 1000, timezone.utc).replace(tzinfo=None)
    return datetime.fromisoformat(str(value).replace("Z", "+00:00")).replace(tzinfo=None)

def encoded_ms(value):
    return int(as_clock(value).replace(tzinfo=timezone.utc).timestamp() * 1000)

def windows(start, stop, days):
    if days <= 0:
        raise ValueError("Window length must be positive")
    start, stop = as_clock(start), as_clock(stop)
    if stop <= start:
        raise ValueError("End date must be later than start date")
    while start < stop:
        end = min(start + timedelta(days=days), stop)
        yield start, end
        start = end

def show_table(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except ImportError:
        for row in rows[:25]:
            print(row)
        if len(rows) > 25:
            print(f"Showing 25 of {len(rows)} rows")

client = AirCastingClient()


## 2b. Test access to your device (`AIRBEAM3:B0B21C7627C4`)

This cell checks whether the AirCasting server returns anything for your device. It tries each spelling of the device ID over the last 365 days, then checks the known session from your map link. It reports server errors separately from empty results, so you can tell "blocked or broken" from "no data uploaded".


In [3]:
def test_device_access(days=365):
    end = datetime.now() + timedelta(days=1)
    start = end - timedelta(days=days)
    results = []
    for package in DEVICE_PACKAGES:
        params = {"start_datetime": start.replace(microsecond=0).isoformat(),
                  "end_datetime": end.replace(microsecond=0).isoformat(),
                  "sensor_package_name": package}
        try:
            response = client.get("/api/v3/sessions", params, refresh=True)
            items = response.get("sessions", []) if isinstance(response, dict) else []
            results.append({"query": package, "result": f"{len(items)} sessions",
                            "session_ids": ", ".join(str(s.get("id")) for s in items[:10])})
        except Exception as error:
            results.append({"query": package, "result": "ERROR", "session_ids": str(error)})
    for sid in KNOWN_SESSION_IDS:
        try:
            info = client.get(f"/api/fixed/sessions/{int(sid)}/streams.json", {"measurements_limit": 0}, refresh=True)
            streams = info.get("streams", [])
            results.append({"query": f"session {sid}", "result": f"OK: '{info.get('title', '')}', {len(streams)} streams",
                            "session_ids": ", ".join(f"{s.get('sensor_name')}={s.get('stream_id')}" for s in streams)})
        except Exception as error:
            results.append({"query": f"session {sid}", "result": "ERROR", "session_ids": str(error)})
    show_table(results)
    if all(r["result"] == "ERROR" for r in results):
        print("Every request failed: check your internet connection or whether aircasting.org is reachable.")
    elif not any(r["result"].startswith(("OK", "1", "2", "3", "4", "5", "6", "7", "8", "9")) for r in results):
        print("The server answered, but nothing matched. Check the device has uploaded data and the session is public.")
    return results

device_test = test_device_access()


,query,result,session_ids
0,AirBeam3:b0b21c7627c4,5 sessions,"1973273, 1973253, 1973290, 1973330, 1973333"
1,AirBeam3:B0B21C7627C4,5 sessions,"1973273, 1973253, 1973290, 1973330, 1973333"
2,AirBeam3-b0b21c7627c4,20 sessions,"1964851, 1959293, 1957893, 1957897, 1954970, 1..."
3,AirBeam3-B0B21C7627C4,20 sessions,"1964851, 1959293, 1957893, 1957897, 1954970, 1..."
4,AIRBEAM3:B0B21C7627C4,0 sessions,
5,session 21374,"OK: 'SB03_3_25', 7 streams","Alphasense-B4-NO=71681, Bosch-BMP180=71682, Se..."


## 3. Optional: find device identifiers from your username

This helper searches publicly contributed **mobile PM2.5 sessions** from the supplied username and returns their device package identifiers. It is not an organisation-wide device list and will not find fixed-only or private-only devices. Set `DEVICE_PACKAGES` using the results, then run discovery. No username means no request.


In [4]:
def lookup_mobile_devices(username, start, stop):
    if not username.strip():
        return []
    q = {"time_from": encoded_ms(start) // 1000,
         "time_to": (encoded_ms(stop) - 1000) // 1000,
         "tags": "", "usernames": username.strip(),
         "west": -180.0, "east": 180.0, "south": -90.0, "north": 90.0,
         "limit": 100, "offset": 0, "sensor_name": "AirBeam-PM2.5",
         "measurement_type": "Particulate Matter", "unit_symbol": "µg/m³"}
    found, previous_pages = {}, set()
    while True:
        result = client.get("/api/mobile/sessions.json", {"q": json.dumps(q)})
        sessions = result.get("sessions") if isinstance(result, dict) else None
        if not isinstance(sessions, list):
            raise RuntimeError("Unexpected mobile discovery response")
        if not sessions:
            if q["offset"] < int(result.get("fetchableSessionsCount", 0)):
                raise RuntimeError("Mobile discovery stopped before the reported session count")
            break
        signature = tuple(str(s.get("id")) for s in sessions)
        if signature in previous_pages:
            raise RuntimeError("Repeated search page; discovery may be incomplete")
        previous_pages.add(signature)
        for session in sessions:
            for stream in session.get("streams", {}).values():
                package = stream.get("sensor_package_name")
                if package:
                    found[package] = {"sensor_package_name": package, "example_session_id": session["id"], "channel": stream.get("sensor_name")}
        q["offset"] += len(sessions)
        total = result.get("fetchableSessionsCount")
        if total is not None and q["offset"] >= int(total):
            break
    return list(found.values())

if LOOKUP_USERNAME.strip():
    device_candidates = lookup_mobile_devices(LOOKUP_USERNAME, DISCOVERY_START, DOWNLOAD_UNTIL)
    show_table(device_candidates)
else:
    print("Optional lookup skipped. Enter your full device identifiers in DEVICE_PACKAGES.")


Optional lookup skipped. Enter your full device identifiers in DEVICE_PACKAGES.


## 4. Discover sessions and measurement streams

Use exact device package strings, including their prefix and separator. Tags alone can discover project sessions, but do not prove hardware identity. A fixed stream's package identifier is not returned by the metadata endpoint used below: exports retain its **device search query** separately from an observed package identifier. Search metadata and failed date windows are saved.


In [5]:
def discover_sessions():
    if not DEVICE_PACKAGES and not PROJECT_TAGS:
        return [], []
    sessions, errors = {}, []
    for package in DEVICE_PACKAGES or [None]:
        for start, stop in windows(DISCOVERY_START, DOWNLOAD_UNTIL, DISCOVERY_WINDOW_DAYS):
            params = {"start_datetime": start.isoformat(), "end_datetime": stop.isoformat()}
            if package:
                params["sensor_package_name"] = package
            if PROJECT_TAGS:
                params["tags[]"] = PROJECT_TAGS
            try:
                response = client.get("/api/v3/sessions", params)
                items = response.get("sessions") if isinstance(response, dict) else None
                if not isinstance(items, list):
                    raise RuntimeError("Unexpected v3 session response")
                for item in items:
                    sid = int(item["id"])
                    record = sessions.setdefault(sid, {**item, "streams": [], "device_queries": []})
                    if package and package not in record["device_queries"]:
                        record["device_queries"].append(package)
                    known = {int(s["id"]) for s in record["streams"]}
                    for stream in item["streams"]:
                        if int(stream["id"]) not in known:
                            record["streams"].append(stream)
                            known.add(int(stream["id"]))
            except Exception as error:
                errors.append({"device_query": package, "start": start.isoformat(), "stop": stop.isoformat(), "error": str(error)})
    return sorted(sessions.values(), key=lambda x: x["id"]), errors

def add_known_sessions(found, errors):
    """Add sessions by numeric ID, e.g. from an aircasting.org map link, when they were not found by device ID."""
    have = {int(s["id"]) for s in found}
    for sid in KNOWN_SESSION_IDS:
        if int(sid) in have:
            continue
        try:
            info = client.get(f"/api/fixed/sessions/{int(sid)}/streams.json", {"measurements_limit": 0})
            start = info.get("start_datetime") or info.get("start_time_local") or info.get("start_time") or DISCOVERY_START
            found.append({"id": int(sid), "type": "FixedSession", "title": info.get("title", ""),
                          "start_datetime": start, "device_queries": [],
                          "streams": [{"id": int(s["stream_id"]), "sensor_name": s.get("sensor_name")} for s in info.get("streams", [])]})
        except Exception as error:
            errors.append({"device_query": f"known_session:{sid}", "error": str(error)})
    return sorted(found, key=lambda x: int(x["id"])), errors

sessions, discovery_errors = discover_sessions()
sessions, discovery_errors = add_known_sessions(sessions, discovery_errors)
save_json(OUTPUT_ROOT / "discovered_sessions.json", sessions)
save_json(OUTPUT_ROOT / "discovery_errors.json", discovery_errors)
show_table([{"session_id": s["id"], "type": s["type"], "start": s["start_datetime"],
             "streams": len(s["streams"]), "device_queries": ";".join(s["device_queries"])} for s in sessions])
print(f"Found {len(sessions)} sessions; {len(discovery_errors)} failed discovery windows.")
if not sessions and not DEVICE_PACKAGES and not PROJECT_TAGS:
    print("Enter your identifiers in the configuration cell and rerun configuration and discovery.")


,session_id,type,start,streams,device_queries
0,21374,FixedSession,1484838039000,7,
1,1954970,FixedSession,2026-02-13T11:40:55,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
2,1955008,FixedSession,2026-02-13T17:09:51,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
3,1955073,FixedSession,2026-02-14T17:37:57,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
4,1955105,FixedSession,2026-02-15T12:56:15,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
5,1955192,FixedSession,2026-02-16T09:23:29,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
6,1955334,FixedSession,2026-02-17T08:46:05,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
7,1956638,FixedSession,2026-02-28T20:00:17,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
8,1957893,FixedSession,2026-03-15T09:45:49,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4
9,1957897,FixedSession,2026-03-15T12:16:20,5,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4


Found 26 sessions; 0 failed discovery windows.


## 5. Region and timestamp handling

GeoJSON must use longitude–latitude coordinates in WGS84. Polygon holes, multiple polygons, and boundary points are supported. If both a rectangle and a polygon are configured, their intersection is used. Missing or withheld coordinates remain in the all-locations export but cannot match a region. No country or county boundary is fabricated.

The notebook preserves every returned observation, including identical repeated observations. API responses do not expose measurement IDs, so timestamp-only deduplication would risk data loss. Each export includes stream identity and a source row index.


In [6]:
def load_region(filename):
    if not filename:
        return []
    obj = json.loads(Path(filename).read_text(encoding="utf-8-sig"))
    if obj.get("crs"):
        name = json.dumps(obj["crs"]).lower()
        if "4326" not in name and "crs84" not in name:
            raise ValueError("Transform the GeoJSON to WGS84 first")
    def polygons(node):
        kind = node.get("type")
        if kind == "FeatureCollection":
            return [p for f in node["features"] for p in polygons(f)]
        if kind == "Feature":
            return polygons(node["geometry"])
        if kind == "Polygon":
            return [node["coordinates"]]
        if kind == "MultiPolygon":
            return node["coordinates"]
        raise ValueError("Use Polygon or MultiPolygon GeoJSON")
    result = polygons(obj)
    if not result:
        raise ValueError("The region contains no polygons")
    for polygon in result:
        if not polygon:
            raise ValueError("Empty polygon")
        for ring in polygon:
            if len(ring) < 4 or ring[0][:2] != ring[-1][:2]:
                raise ValueError("Each polygon ring must be closed with at least four positions")
            for point in ring:
                x, y = point[:2]
                if not (-180 <= x <= 180 and -90 <= y <= 90):
                    raise ValueError("Invalid longitude/latitude coordinates")
            if any(abs(a[0] - b[0]) > 180 for a, b in zip(ring, ring[1:])):
                raise ValueError("Split regions at the antimeridian before use")
    return result

def ring_position(x, y, ring):
    inside = False
    for a, b in zip(ring, ring[1:]):
        ax, ay = a[:2]; bx, by = b[:2]
        cross = (x - ax) * (by - ay) - (y - ay) * (bx - ax)
        if abs(cross) <= 1e-12 and min(ax, bx) - 1e-12 <= x <= max(ax, bx) + 1e-12 and min(ay, by) - 1e-12 <= y <= max(ay, by) + 1e-12:
            return 2  # boundary
        if (ay > y) != (by > y) and x < (bx - ax) * (y - ay) / (by - ay) + ax:
            inside = not inside
    return 1 if inside else 0

def polygon_covers(x, y, polygon):
    outer = ring_position(x, y, polygon[0])
    if outer == 0:
        return False
    if outer == 2:
        return True
    for hole in polygon[1:]:
        position = ring_position(x, y, hole)
        if position == 2:
            return True
        if position == 1:
            return False
    return True

def valid_coordinates(lon, lat):
    try:
        return math.isfinite(float(lon)) and math.isfinite(float(lat)) and -180 <= float(lon) <= 180 and -90 <= float(lat) <= 90
    except (TypeError, ValueError):
        return False

def region_matches(lon, lat, polygons):
    if BBOX is None and not polygons:
        return True
    if not valid_coordinates(lon, lat):
        return False
    x, y = float(lon), float(lat)
    if BBOX is not None:
        west, south, east, north = BBOX
        if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
            raise ValueError("BBOX must be (west, south, east, north), without antimeridian crossing")
        if not (west <= x <= east and south <= y <= north):
            return False
    return not polygons or any(polygon_covers(x, y, p) for p in polygons)

def timestamp_fields(raw):
    try:
        clock = as_clock(raw)
    except (ValueError, TypeError, OverflowError, OSError):
        return "", "", "invalid_source_time"
    if TIME_CONVENTION == "unverified":
        return clock.isoformat(), "", "source_clock_unverified"
    if TIME_CONVENTION == "utc":
        instant = clock.replace(tzinfo=timezone.utc) if isinstance(raw, (int, float)) else datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
        if instant.tzinfo is None:
            instant = instant.replace(tzinfo=timezone.utc)
        return clock.isoformat(), instant.astimezone(timezone.utc).isoformat(), "UTC_assumption_confirmed_by_user"
    if TIME_CONVENTION != "local_as_utc":
        raise ValueError("Unknown TIME_CONVENTION")
    zone = ZoneInfo(TIMEZONE_NAME)
    candidates = set()
    for fold in (0, 1):
        utc = clock.replace(tzinfo=zone, fold=fold).astimezone(timezone.utc)
        if utc.astimezone(zone).replace(tzinfo=None) == clock:
            candidates.add(utc.isoformat())
    if len(candidates) != 1:
        return clock.isoformat(), "", "DST_ambiguous" if candidates else "DST_nonexistent"
    return clock.isoformat(), next(iter(candidates)), "local_clock_zone_confirmed_by_user"

region_polygons = load_region(REGION_GEOJSON)
region_matches(0, 0, region_polygons)  # validate rectangle configuration
if TIME_CONVENTION not in {"unverified", "utc", "local_as_utc"}:
    raise ValueError("Unknown TIME_CONVENTION")
if TIME_CONVENTION == "local_as_utc":
    ZoneInfo(TIMEZONE_NAME)  # validate the timezone before downloading


## 6. Retrieve complete mobile streams and fixed measurements

Mobile stream details include measurements and actual package names. Their stream IDs are checked against discovery. For fixed recordings, metadata is requested with `measurements_limit=0`, then the dedicated fixed-measurement endpoint is queried in bounded time windows. Its default display endpoint contains only recent measurements and is not used as a historical export.

Different mobile channels are fetched separately to retain stream identity. Source responses remain in `raw_responses`. A failed stream/window is recorded rather than silently treated as an empty successful download. The fixed API exposes no total measurement count, so completeness cannot be independently guaranteed.


In [7]:
def normalise_package(value):
    parts = re.split(r"([:\-])", str(value), maxsplit=1)
    return parts[0] + parts[1] + parts[2].lower() if len(parts) == 3 else str(value)

def measurement_batches(session):
    sid = int(session["id"])
    if session["type"] == "MobileSession":
        for stream in session["streams"]:
            stream_id = int(stream["id"])
            try:
                result = client.get(f"/api/mobile/sessions2/{sid}.json", {"sensor_name": stream["sensor_name"]})
                candidates = result.get("streams", {})
                matches = [s for s in candidates.values() if int(s["id"]) == stream_id]
                if len(matches) != 1:
                    raise RuntimeError("Returned stream ID differs from discovery; ambiguous channel mapping")
                metadata = matches[0]
                package = metadata.get("sensor_package_name")
                queries = session.get("device_queries", [])
                if queries and (not package or normalise_package(package) not in {normalise_package(q) for q in queries}):
                    yield stream_id, {}, [], {"status": "excluded_other_device", "session_id": sid, "stream_id": stream_id}
                    continue
                records = metadata.get("measurements")
                if not isinstance(records, list):
                    raise RuntimeError("Missing mobile measurement array")
                expected = metadata.get("measurements_count")
                status = "count_matches" if expected is not None and int(expected) == len(records) else "count_unavailable" if expected is None else "count_mismatch"
                yield stream_id, metadata, records, {"status": status, "session_id": sid, "stream_id": stream_id, "source_count": expected, "returned_count": len(records)}
            except Exception as error:
                yield stream_id, {}, [], {"status": "failed", "session_id": sid, "stream_id": stream_id, "error": str(error)}
    elif session["type"] == "FixedSession":
        try:
            info = client.get(f"/api/fixed/sessions/{sid}/streams.json", {"measurements_limit": 0})
            metadata_by_id = {int(s["stream_id"]): s for s in info["streams"]}
        except Exception as error:
            yield None, {}, [], {"status": "failed", "session_id": sid, "error": str(error)}
            return
        for stream in session["streams"]:
            stream_id = int(stream["id"])
            if stream_id not in metadata_by_id:
                yield stream_id, {}, [], {"status": "failed", "session_id": sid, "stream_id": stream_id, "error": "Fixed stream metadata missing"}
                continue
            metadata = {**metadata_by_id[stream_id], "latitude": None if info.get("is_indoor") else info.get("latitude"),
                        "longitude": None if info.get("is_indoor") else info.get("longitude"), "fixed": True,
                        "is_indoor": info.get("is_indoor", False)}
            begin = max(as_clock(DOWNLOAD_FROM), as_clock(session["start_datetime"]))
            stop = as_clock(DOWNLOAD_UNTIL)
            if begin >= stop:
                continue
            for start, end in windows(begin, stop, FIXED_WINDOW_DAYS):
                report = {"session_id": sid, "stream_id": stream_id, "start": start.isoformat(), "stop_exclusive": end.isoformat()}
                try:
                    records = client.get("/api/v3/fixed_measurements", {"stream_id": str(stream_id), "start_time": encoded_ms(start), "end_time": encoded_ms(end) - 1})
                    if not isinstance(records, list):
                        raise RuntimeError("Missing fixed measurement array")
                    if any(not (start <= as_clock(m["time"]) < end) for m in records):
                        raise RuntimeError("Fixed endpoint returned a record outside its requested window")
                    yield stream_id, metadata, records, {**report, "status": "retrieved_count_unverified", "returned_count": len(records)}
                except Exception as error:
                    yield stream_id, {}, [], {**report, "status": "failed", "error": str(error)}
    else:
        yield None, {}, [], {"status": "failed", "session_id": sid, "error": "Unsupported session type: " + str(session["type"])}


## 7. Download and export

This cell retrieves every discovered stream. `all_recordings.csv` contains all returned readings within your chosen dates, across all locations. `selected_region.csv` and `by_device/*.csv` use the same region filter. Device query groups without observed hardware identifiers are explicitly labelled as query groups. `manifest.json` records counts, failures, filter geometry, and retrieval provenance. The ZIP includes the CSV exports and manifest; raw responses stay outside the ZIP in the cache folder.

For subsequent downloads, rerun discovery and this cell with `REFRESH=True`. This deliberately rechecks old sessions, so late uploads are not missed solely because their measurement dates are old. It is not a background scheduler.


In [8]:
COLUMNS = ["device_group", "sensor_package_name", "device_query", "session_id", "stream_id", "session_type",
           "sensor_name", "measurement_type", "unit", "raw_time", "source_time", "time_utc", "time_status",
           "value", "latitude", "longitude", "coordinate_source", "source_row_index"]

def export_recordings(session_list, errors):
    run = OUTPUT_ROOT / ("export_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    run.mkdir(parents=True)
    (run / "by_device").mkdir()
    reports, counts, device_files = [], Counter(), {}
    start, stop = as_clock(DOWNLOAD_FROM), as_clock(DOWNLOAD_UNTIL)
    if stop <= start:
        raise ValueError("DOWNLOAD_UNTIL must follow DOWNLOAD_FROM")
    source_indices = Counter()
    selected_preview = []
    call_start = len(client.calls)
    try:
        with (run / "all_recordings.csv").open("w", newline="", encoding="utf-8-sig") as all_f, (run / "selected_region.csv").open("w", newline="", encoding="utf-8-sig") as selected_f:
            all_writer, selected_writer = [csv.DictWriter(f, fieldnames=COLUMNS) for f in (all_f, selected_f)]
            all_writer.writeheader(); selected_writer.writeheader()
            for position, session in enumerate(session_list, 1):
                print(f"Session {position}/{len(session_list)}: {session['id']}")
                for stream_id, metadata, records, report in measurement_batches(session):
                    reports.append(report)
                    for reading in records:
                        key = (session["id"], stream_id)
                        source_indices[key] += 1
                        counts["source_records_returned"] += 1
                        source_time, utc_time, time_status = timestamp_fields(reading.get("time"))
                        if not source_time:
                            counts["invalid_timestamp_records"] += 1
                            continue
                        if not (start <= as_clock(source_time) < stop):
                            counts["outside_date_range"] += 1
                            continue
                        package = metadata.get("sensor_package_name", "")
                        queries = ";".join(session.get("device_queries", []))
                        group = package or ("query:" + queries if queries else "session:" + str(session["id"]))
                        fixed = metadata.get("fixed", False)
                        lat = metadata.get("latitude") if fixed else reading.get("latitude")
                        lon = metadata.get("longitude") if fixed else reading.get("longitude")
                        coordinate_source = "withheld_indoor" if metadata.get("is_indoor") else "fixed_session" if fixed else "measurement"
                        row = {"device_group": group, "sensor_package_name": package, "device_query": queries,
                               "session_id": session["id"], "stream_id": stream_id, "session_type": session["type"],
                               "sensor_name": metadata.get("sensor_name", ""), "measurement_type": metadata.get("measurement_type", ""),
                               "unit": metadata.get("unit_symbol", metadata.get("sensor_unit", "")),
                               "raw_time": reading.get("time"), "source_time": source_time, "time_utc": utc_time, "time_status": time_status,
                               "value": reading.get("value"), "latitude": lat, "longitude": lon,
                               "coordinate_source": coordinate_source, "source_row_index": source_indices[key]}
                        all_writer.writerow(row)
                        counts["all_recordings_rows"] += 1
                        if not valid_coordinates(lon, lat):
                            counts["missing_or_invalid_coordinates"] += 1
                        if region_matches(lon, lat, region_polygons):
                            selected_writer.writerow(row)
                            counts["selected_region_rows"] += 1
                            if group not in device_files:
                                file = (run / "by_device" / (safe_name(group) + ".csv")).open("w", newline="", encoding="utf-8-sig")
                                writer = csv.DictWriter(file, fieldnames=COLUMNS)
                                writer.writeheader()
                                device_files[group] = (file, writer)
                            device_files[group][1].writerow(row)
                            if len(selected_preview) < 1000:
                                selected_preview.append(row)
    finally:
        for file, writer in device_files.values():
            file.close()
    problem = bool(errors) or counts["invalid_timestamp_records"] > 0 or any(r["status"] in {"failed", "count_mismatch"} for r in reports)
    manifest = {"export_time_utc": datetime.now(timezone.utc).isoformat(), "source": BASE_URL,
                "status": "partial_or_inconsistent" if problem else "requests_completed_upstream_completeness_unverified",
                "scope": {"device_packages": DEVICE_PACKAGES, "project_tags": PROJECT_TAGS, "discovery_start": DISCOVERY_START,
                          "download_from": DOWNLOAD_FROM, "download_until_exclusive": DOWNLOAD_UNTIL,
                          "region_name": REGION_NAME, "bbox": BBOX, "geojson_polygons": region_polygons,
                          "time_convention": TIME_CONVENTION, "timezone": TIMEZONE_NAME},
                "counts": dict(counts), "sessions_discovered": len(session_list), "discovery_errors": errors,
                "retrieval_reports": reports, "requests": client.calls[call_start:], "refresh": REFRESH,
                "notes": ["No measurement IDs or fixed-series total counts are exposed by these endpoints.",
                          "Device queries are search associations, not independently observed fixed hardware identifiers.",
                          "Raw responses and metadata are retained separately under raw_responses."]}
    save_json(run / "manifest.json", manifest)
    save_json(run / "sessions.json", session_list)
    archive = run / "recordings.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
        for path in sorted(run.rglob("*")):
            if path.is_file() and path != archive:
                zipped.write(path, path.relative_to(run))
    return run, manifest, selected_preview

if sessions:
    export_directory, export_manifest, preview_rows = export_recordings(sessions, discovery_errors)
    print(json.dumps({"status": export_manifest["status"], "counts": export_manifest["counts"]}, indent=2))
    from IPython.display import FileLink, display
    for filename in ["all_recordings.csv", "selected_region.csv", "recordings.zip", "manifest.json"]:
        display(FileLink(str(export_directory / filename)))
else:
    preview_rows = []
    print("No sessions ready to download. Check configuration and discovery_errors.json; no data has been fabricated.")


Session 1/26: 21374
Session 2/26: 1954970
Session 3/26: 1955008
Session 4/26: 1955073
Session 5/26: 1955105
Session 6/26: 1955192
Session 7/26: 1955334
Session 8/26: 1956638
Session 9/26: 1957893
Session 10/26: 1957897
Session 11/26: 1959166
Session 12/26: 1959185
Session 13/26: 1959186
Session 14/26: 1959293
Session 15/26: 1959522
Session 16/26: 1959559
Session 17/26: 1959569
Session 18/26: 1959570
Session 19/26: 1959591
Session 20/26: 1964822
Session 21/26: 1964851
Session 22/26: 1973253
Session 23/26: 1973273
Session 24/26: 1973290
Session 25/26: 1973330
Session 26/26: 1973333
{
  "status": "requests_completed_upstream_completeness_unverified",
  "counts": {
    "source_records_returned": 102770,
    "all_recordings_rows": 102770,
    "missing_or_invalid_coordinates": 21720,
    "selected_region_rows": 102770
  }
}


C:\Users\0135816s\Downloads\aircasting_downloads\export_20260915T200010_026336Z\all_recordings.csv

C:\Users\0135816s\Downloads\aircasting_downloads\export_20260915T200010_026336Z\selected_region.csv

C:\Users\0135816s\Downloads\aircasting_downloads\export_20260915T200010_026336Z\recordings.zip

C:\Users\0135816s\Downloads\aircasting_downloads\export_20260915T200010_026336Z\manifest.json

## 8. Optional preview and draw a region

The table and map show at most 1,000 selected readings; exports contain all matching rows. Map tiles require an internet connection. If folium is unavailable, the CSV workflow still works; install it with `%pip install folium` in a separate cell. Draw a polygon/rectangle and click **Export**, then set `REGION_GEOJSON` to that downloaded file's path and rerun the region and export cells. A drawing does not change the filter until loaded.


In [9]:
if preview_rows:
    show_table(preview_rows[:20])
try:
    import folium
    from folium.plugins import Draw, FastMarkerCluster
    map_view = folium.Map(location=[54.95, -7.9], zoom_start=8)
    Draw(export=True, filename="my_region.geojson", draw_options={"polyline": False, "circle": False, "circlemarker": False, "marker": False}).add_to(map_view)
    points = [[float(r["latitude"]), float(r["longitude"])] for r in preview_rows if valid_coordinates(r["longitude"], r["latitude"])]
    if points:
        FastMarkerCluster(points).add_to(map_view)
    if region_polygons:
        folium.GeoJson({"type": "MultiPolygon", "coordinates": region_polygons}, name="Selected region").add_to(map_view)
    display(map_view)
except ImportError:
    print("Optional map skipped. CSV downloads do not require folium.")


,device_group,sensor_package_name,device_query,session_id,stream_id,session_type,sensor_name,measurement_type,unit,raw_time,source_time,time_utc,time_status,value,latitude,longitude,coordinate_source,source_row_index
0,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770982922000,2026-02-13T11:42:02,,source_clock_unverified,68.0,None,None,withheld_indoor,1
1,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770982884000,2026-02-13T11:41:24,,source_clock_unverified,66.0,None,None,withheld_indoor,2
2,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770982982000,2026-02-13T11:43:02,,source_clock_unverified,69.0,None,None,withheld_indoor,3
3,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983102000,2026-02-13T11:45:02,,source_clock_unverified,71.0,None,None,withheld_indoor,4
4,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983042000,2026-02-13T11:44:02,,source_clock_unverified,70.0,None,None,withheld_indoor,5
5,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983162000,2026-02-13T11:46:02,,source_clock_unverified,71.0,None,None,withheld_indoor,6
6,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983222000,2026-02-13T11:47:02,,source_clock_unverified,72.0,None,None,withheld_indoor,7
7,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983282000,2026-02-13T11:48:02,,source_clock_unverified,72.0,None,None,withheld_indoor,8
8,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983342000,2026-02-13T11:49:02,,source_clock_unverified,73.0,None,None,withheld_indoor,9
9,query:AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,,AirBeam3-b0b21c7627c4;AirBeam3-B0B21C7627C4,1954970,2860941,FixedSession,AirBeam3-F,Temperature,F,1770983402000,2026-02-13T11:50:02,,source_clock_unverified,73.0,None,None,withheld_indoor,10


Optional map skipped. CSV downloads do not require folium.


## 9. Optional: a known session belonging to an authenticated account

The source implements HTTP Basic authentication with the **AirCasting authentication token as the username and `X` as the password**. This is not an ordinary account password and not an organisation-wide credential. The account endpoint restricts access to that account's own session. Obtain the token through the service's supported process; do not share participant passwords.

The function below saves one known **numeric session ID** and all exposed stream measurements as a raw JSON file. It intentionally does not call the app's POST sync routes, which can change server state. Its output is separate from the public CSV workflow. It does not enumerate private sessions or unlock other participants' accounts.


In [10]:
def download_owned_session(session_id):
    from getpass import getpass
    if not str(session_id).isdigit():
        raise ValueError("Use a numeric session ID for this endpoint")
    token = os.environ.get("AIRCASTING_API_TOKEN") or getpass("AirCasting authentication token (hidden): ")
    if not token:
        raise ValueError("A token is required")
    result = client.get(f"/api/user/sessions/{int(session_id)}.json", {"stream_measurements": "true"}, token=token, refresh=True)
    destination = OUTPUT_ROOT / "account_exports" / f"session_{int(session_id)}.json"
    save_json(destination, result)
    return destination

# To use with your own known session:
# account_file = download_owned_session(YOUR_NUMERIC_SESSION_ID)
# display(FileLink(str(account_file)))


## Endpoint evidence and practical limits

These references are relative to the supplied `AirCasting-master` directory. [Hosted API documentation](https://aircasting.org/api-docs) may reflect a different revision.

| Purpose | Endpoint / source evidence |
|---|---|
| Device/tag session discovery | `GET /api/v3/sessions`; `app/services/sessions/contract.rb`, `app/repositories/sessions_repository.rb`, `app/serializers/sessions_serializer.rb` |
| Paginated mobile username lookup | `GET /api/mobile/sessions.json`; `app/models/api/mobile_sessions_contract.rb`, `app/services/api/to_mobile_sessions_array.rb` |
| Full mobile stream and package metadata | `GET /api/mobile/sessions2/{id}.json?sensor_name=...`; `app/controllers/api/mobile/sessions_controller.rb`, `app/services/api/to_session_hash2.rb` |
| Fixed stream metadata | `GET /api/fixed/sessions/{id}/streams.json?measurements_limit=0`; `app/models/api/session_contract.rb`, `app/services/api/to_fixed_session_with_streams_hash.rb` |
| Historical fixed measurements | `GET /api/v3/fixed_measurements`; `app/models/api/fixed_measurements_contract.rb`, `app/repositories/fixed_measurements_repository.rb` |
| Time convention | `spec/swagger/v3/measurements_spec.rb` explicitly describes local-as-UTC; `app/services/api/to_session_hash2.rb` formats wall clocks with a Z suffix |
| Account-only session | `app/controllers/api/base_controller.rb`, `app/controllers/api/user_sessions_controller.rb`, `config/routes.rb` |

**No results:** verify the complete package string, deployment start, uploads, and sharing/access settings. `/api/sensors` is a catalogue of measurement types, not a registry of your physical devices. A region containing unrelated public sensors does not make them part of your project.

**HTTP errors:** inspect the endpoint and access requirements. A 404 or non-JSON response may indicate the hosted revision differs from this ZIP. Failure details appear in `discovery_errors.json` and the export manifest.

**Coverage:** this notebook retrieves API-exposed readings and does not claim to recover unsynced, deleted, or inaccessible data. Mobile count mismatches and failed fixed windows are reported. Live API completeness cannot be proven without a known source session/export for comparison. Very large individual mobile sessions are returned as a complete JSON response and require sufficient memory.

**Geography:** use valid, non-self-intersecting GeoJSON rings. Administrative boundaries must be supplied from your chosen authoritative source. Indoor fixed coordinates are treated as withheld. Exact positions and timestamps remain sensitive even when participant names are omitted.
